# アメダス観測データ 取得 → 変換 → push（Colab完結版）

`landfallJP=true`または`damaging=true`な台風（従来の291台風）の
AMeDAS観測データは取得済みです。`fetch_tydb_damage.py --all`で
TYDB被害データのカバレッジを広げた場合、`scripts/list_damage_gaps.py
--kind obs` がその中で風または雨のAMeDAS観測データがまだ無い台風を
一覧してくれます。

このノートブックは、その一覧の台風のAMeDAS風・雨観測データを、
**すべてColab上だけで**取得（気象庁obsdl API）→ 台風ごとのJSONに変換
→ GitHubへpush します。手元PC・手動でのGoogle Driveアップロードは
不要です（`scripts/fetch_amedas_obsdl.py` は`requests`だけで完結する
スクリプトで、Colabの実行環境からも気象庁サイトへ到達できます）。
既に取得済みの台風は自動でスキップされるので、このノートブックを
再実行するだけで追加分だけが処理されます。

**取得キャッシュはGoogle Drive上に置く**ので、Colabのセッションが切断
されても、ノートブックを再実行すれば完了済みの分はスキップされ、続きから
再開できます。

## 使い方
①→②→③→④→⑤→⑥の順に上から実行してください。

## ① Google Driveをマウント（取得キャッシュの保存先）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_RAW_DIR = pathlib.Path('/content/drive/MyDrive/typhoon_raw_amedas')
DRIVE_RAW_DIR.mkdir(parents=True, exist_ok=True)
print('cache dir:', DRIVE_RAW_DIR)

## ② GitHubへのPersonal Access Token（PAT）を用意

すでに持っていれば③に進んでOKです。まだの場合:

1. GitHubの https://github.com/settings/tokens?type=beta を開く
2. **Generate new token** → このリポジトリ（`awg-yk/typhoon-wind-rainfall`）
   に対して **Contents: Read and write** 権限を付与
3. 発行されたトークン（`github_pat_...` から始まる文字列）をコピー

次のセルを実行すると入力欄が出るので、そこに貼り付けてください
（画面には表示されず、Colab上にも保存されません）。

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass('GitHubのPersonal Access Tokenを貼り付けてEnter: ')

## ③ リポジトリを取得し、raw_amedasをDrive上のキャッシュにリンク

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/remaining-tasks-gzid2a'
REMOTE_URL = f'https://{GITHUB_TOKEN}@github.com/awg-yk/typhoon-wind-rainfall.git'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REMOTE_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!cd {REPO_DIR} && git config user.email "colab@example.com"
!cd {REPO_DIR} && git config user.name "Colab"

# data/raw_amedas をDrive上のキャッシュフォルダへのシンボリックリンクにする
# -- こうしておくとColabのセッションが切れてもバッチ単位の取得結果が
# 消えず、ノートブックを再実行するだけで続きから再開できる
raw_amedas_path = f'{REPO_DIR}/data/raw_amedas'
if os.path.islink(raw_amedas_path) or os.path.isdir(raw_amedas_path):
    !rm -rf {raw_amedas_path}
!ln -s {DRIVE_RAW_DIR} {raw_amedas_path}
print('linked', raw_amedas_path, '->', DRIVE_RAW_DIR)

## ④ 取得（気象庁obsdl APIから風・雨のCSVをダウンロード）

`scripts/list_damage_gaps.py --kind obs` が、TYDB被害データはあるのに
風または雨のAMeDAS観測データが欠けている台風コードを一覧してくれる
ので、それをそのまま`--codes`に渡します（`data/tydb_damage/`を
`fetch_tydb_damage.py --all`で広げた後に実行する想定）。
まだ `data/raw_amedas/<コード>_{wind,rain}.csv` が無いものだけを
取得します（既存分は自動スキップ）。

リクエスト間隔は既定3秒（`--sleep`で変更可、下のセルは1秒に短縮
済み）です。obsdl側に明記されたレート制限は無いため、体感で失敗
（`FAILED`やリトライ）が増えなければ、さらに `--sleep 0.5` などへ
縮めても大きな問題はなさそうです。逆に失敗が増えるようなら3秒程度に
戻してください。既に完了したバッチはキャッシュされているので、
`--sleep`の値を変えて再実行しても続きから進みます。

In [ ]:
!pip install -q requests
!cd {REPO_DIR} && python3 scripts/fetch_amedas_obsdl.py --codes $(python3 scripts/list_damage_gaps.py --kind obs) --kind wind,rain --sleep 1.0

## ⑤ 変換実行

`data/raw_amedas/*_{wind,rain}.csv` を台風ごとの
`data/storms_obs/<台風コード>_{wind,rain}.json` に変換します。
④が全て完了していなくても、その時点で揃っているCSVだけ変換されます
（未完了分は次回このセルを再実行すれば追加で変換されます）。

In [ ]:
!cd {REPO_DIR} && python3 scripts/convert_amedas_csv.py --all

import pathlib
out_dir = pathlib.Path(REPO_DIR) / 'data' / 'storms_obs'
files = list(out_dir.glob('*.json'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} JSON files, {total_mb:.1f} MB total')

## ⑥ GitHubへコミット & push

`data/storms_obs/` 配下のJSONだけをコミットします
（`git status` の出力で他のファイルが混ざっていないか一応確認してください）。

In [ ]:
!cd {REPO_DIR} && git add data/storms_obs/
!cd {REPO_DIR} && git status --short
!cd {REPO_DIR} && git commit -m "Add AMeDAS wind/rain observation data (data/storms_obs/*.json)"
!cd {REPO_DIR} && git push origin {BRANCH}

## 完了後

- GitHub上の該当ブランチに `data/storms_obs/*.json` が反映されていれば
  成功です。まだ全台風分に届いていない場合は、④→⑤→⑥をもう一度
  実行すれば続きが処理されます（④は完了済み分をスキップするので
  何度実行しても安全です）。
- フロントエンド（`index.html`）は、その台風に `data/storms_obs/<コード>_
  {wind,rain}.json` があれば自動的にAMeDAS観測網（風976地点・雨1669地点）で
  表示します（統合済み）。特別な追加作業は不要です。年/台風の絞り込み
  チェックボックスは上陸・通過台風だけを対象にしているので、被害台風は
  チェックを外した状態で年から選べば表示されます。
- このノートブックのセッションを閉じれば、貼り付けたトークンはColab上から
  消えます。念のため、使い終わったトークンはGitHubの設定画面から失効させて
  おくとより安全です。
- Drive上の `MyDrive/typhoon_raw_amedas` フォルダには取得済みCSVが
  残ります。全台風分の変換・pushが終わったら、容量整理のために削除して
  構いません。